**Обучение языковой модели с помощью LSTM**

В этом задании Вам предстоит обучить языковую модель с помощью рекуррентной нейронной сети.

In [1]:
!pip install -qU datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 19.7 MB/s eta 0:00:00


Импорт необходимых библиотек

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from datasets import load_dataset
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.model_selection import train_test_split
import nltk

from collections import Counter
from typing import List

import seaborn
seaborn.set(palette='summer')

In [3]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

## Подготовка данных

Воспользуемся датасетом imdb. В нем хранятся отзывы о фильмах с сайта imdb. Загрузим данные с помощью функции ```load_dataset```

In [5]:
# Загрузим датасет
dataset = load_dataset('imdb')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

### Препроцессинг данных и создание словаря

Далее вам необходмо самостоятельно произвести препроцессинг данных и получить словарь или же просто ```set``` строк. Что необходимо сделать:

1. Разделить отдельные тренировочные примеры на отдельные предложения с помощью функции ```sent_tokenize``` из бибилиотеки ```nltk```. Каждое отдельное предложение будет одним тренировочным примером.
2. Оставить только те предложения, в которых меньше ```word_threshold``` слов.
3. Посчитать частоту вхождения каждого слова в оставшихся предложениях. Для деления предлоения на отдельные слова удобно использовать функцию ```word_tokenize```.
4. Создать объект ```vocab``` класса ```set```, положить в него служебные токены '\<unk\>', '\<bos\>', '\<eos\>', '\<pad\>' и vocab_size самых частовстречающихся слов.   

In [6]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [7]:
sentences = []
word_threshold = 32

# Получить отдельные предложения и поместить их в sentences
for example in tqdm(dataset['train']):
    text = example['text']
    for sent in sent_tokenize(text):
        tokens = word_tokenize(sent)
        if 0 < len(tokens) <= word_threshold:
            sentences.append(sent.lower())

  0%|          | 0/25000 [00:00<?, ?it/s]

In [8]:
print("Всего предложений:", len(sentences))

Всего предложений: 202657


Посчитаем для каждого слова его встречаемость.

In [9]:
words = Counter()

# Расчет встречаемости слов
for sent in tqdm(sentences):
    tokens = word_tokenize(sent)
    words.update(tokens)

  0%|          | 0/202657 [00:00<?, ?it/s]

In [10]:
sorted(words.items(), key=lambda x: x[1], reverse=True)[:20], len(words)

([('.', 173698),
  ('the', 157414),
  (',', 119240),
  ('a', 78231),
  ('and', 76172),
  ('of', 65489),
  ('to', 63011),
  ('is', 59361),
  ('it', 53502),
  ('i', 50452),
  ('this', 43907),
  ('in', 43009),
  ('that', 34636),
  ("'s", 30364),
  ('was', 28597),
  ('movie', 25327),
  ('for', 21676),
  ('but', 21233),
  ('as', 20577),
  ('film', 20035)],
 69902)

Добавим в словарь ```vocab_size``` самых встречающихся слов.

In [11]:
vocab = set()
vocab_size = 40000

# Наполнение словаря
tokens_and_words = ['<unk>', '<bos>', '<eos>', '<pad>'] + [w for w, c in words.most_common(vocab_size)]
vocab.update(tokens_and_words)

In [12]:
assert '<unk>' in vocab
assert '<bos>' in vocab
assert '<eos>' in vocab
assert '<pad>' in vocab
assert len(vocab) == vocab_size + 4

In [13]:
print("Всего слов в словаре:", len(vocab))

Всего слов в словаре: 40004


### Подготовка датасета

Далее, как и в семинарском занятии, подготовим датасеты и даталоадеры.

В классе ```WordDataset``` вам необходимо реализовать метод ```__getitem__```, который будет возвращать сэмпл данных по входному idx, то есть список целых чисел (индексов слов).

Внутри этого метода необходимо добавить служебные токены начала и конца последовательности, а также токенизировать соответствующее предложение с помощью ```word_tokenize``` и сопоставить ему индексы из ```word2ind```.

In [14]:
word2ind = {char: i for i, char in enumerate(vocab)}
ind2word = {i: char for char, i in word2ind.items()}

In [15]:
class WordDataset:
    def __init__(self, sentences):
        self.data = sentences
        self.unk_id = word2ind['<unk>']
        self.bos_id = word2ind['<bos>']
        self.eos_id = word2ind['<eos>']
        self.pad_id = word2ind['<pad>']

    def __getitem__(self, idx: int) -> List[int]:
        tokenized_sentence = [self.bos_id]
        # Допишите код здесь
        sent = self.data[idx]
        tokens = word_tokenize(sent)

        for token in tokens:
            token_id = word2ind.get(token, self.unk_id)
            tokenized_sentence.append(token_id)
        tokenized_sentence.append(self.eos_id)

        return tokenized_sentence

    def __len__(self) -> int:
        return len(self.data)

In [16]:
def collate_fn_with_padding(
    input_batch: List[List[int]], pad_id=word2ind['<pad>']) -> torch.Tensor:
    seq_lens = [len(x) for x in input_batch]
    max_seq_len = max(seq_lens)

    new_batch = []
    for sequence in input_batch:
        for _ in range(max_seq_len - len(sequence)):
            sequence.append(pad_id)
        new_batch.append(sequence)

    sequences = torch.LongTensor(new_batch).to(device)

    new_batch = {
        'input_ids': sequences[:,:-1],
        'target_ids': sequences[:,1:]
    }

    return new_batch

In [17]:
# train_sentences, eval_sentences = train_test_split(sentences, test_size=0.2)
# eval_sentences, test_sentences = train_test_split(sentences, test_size=0.5)

train_sentences, eval_test_sentences = train_test_split(sentences, test_size=0.2)
eval_sentences, test_sentences = train_test_split(eval_test_sentences, test_size=0.5)

train_dataset = WordDataset(train_sentences)
eval_dataset = WordDataset(eval_sentences)
test_dataset = WordDataset(test_sentences)

batch_size = 128

train_dataloader = DataLoader(
    train_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

eval_dataloader = DataLoader(
    eval_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

test_dataloader = DataLoader(
    test_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

## Обучение и архитектура модели

Вам необходимо на практике проверить, что влияет на качество языковых моделей. В этом задании нужно провести серию экспериментов с различными вариантами языковых моделей и сравнить различия в конечной перплексии на тестовом множестве.

Возможные идеи для экспериментов:

* Различные RNN-блоки, например, LSTM или GRU. Также можно добавить сразу несколько RNN блоков друг над другом с помощью аргумента num_layers. Вам поможет официальная документация [здесь](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)
* Различные размеры скрытого состояния. Различное количество линейных слоев после RNN-блока. Различные функции активации.
* Добавление нормализаций в виде Dropout, BatchNorm или LayerNorm
* Различные аргументы для оптимизации, например, подбор оптимального learning rate или тип алгоритма оптимизации SGD, Adam, RMSProp и другие
* Любые другие идеи и подходы

После проведения экспериментов необходимо составить таблицу результатов, в которой описан каждый эксперимент и посчитана перплексия на тестовом множестве.

Учтите, что эксперименты, которые различаются, например, только размером скрытого состояния или количеством линейных слоев считаются, как один эксперимент.

### Функция evaluate

Заполните функцию ```evaluate```

In [18]:
def evaluate(model, criterion, dataloader) -> float:
    model.eval()
    perplexity = []
    with torch.no_grad():
        for batch in dataloader:
            # Посчитайте логиты предсказаний следующих слов
            logits = model(batch['input_ids'])

            loss = criterion(logits, batch['target_ids'].flatten())
            perplexity.append(torch.exp(loss).item())

    perplexity = sum(perplexity) / len(perplexity)

    return perplexity

### Train loop

Напишите функцию для обучения модели.

In [19]:
# Напишите код здесь
def train_model(model, train_dataloader, eval_dataloader,
                optimizer, criterion, num_epochs):
    model.to(device)
    best_eval_ppl = float('inf')
    best_state_dict = None

    train_losses = []
    eval_ppls = []

    for epoch in range(num_epochs):
        model.train()
        epoch_losses = []

        for batch in tqdm(train_dataloader, desc=f'Epoch {epoch+1}/{num_epochs}'):
            input_ids = batch['input_ids']
            target_ids = batch['target_ids'].flatten()

            optimizer.zero_grad()

            logits = model(input_ids)
            loss = criterion(logits, target_ids)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_losses.append(loss.item())

        avg_train_loss = sum(epoch_losses) / len(epoch_losses)
        train_losses.append(avg_train_loss)

        eval_ppl = evaluate(model, criterion, eval_dataloader)
        eval_ppls.append(eval_ppl)

        print(f'Epoch {epoch+1}: train_loss={avg_train_loss:.3f}, eval_ppl={eval_ppl:.3f}')

        if eval_ppl < best_eval_ppl:
            best_eval_ppl = eval_ppl
            best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state_dict)

    return train_losses, eval_ppls, best_eval_ppl

### Первый эксперимент

Определите архитектуру модели и обучите её.

In [20]:
class LanguageModel(nn.Module):
    def __init__(self, vocab_size=len(vocab), embedding_dim=128,
                 hidden_size=256, num_layers=1, dropout=0.2):
        super().__init__()
        # Опишите свою нейронную сеть здесь
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=word2ind['<pad>'])

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        # А тут опишите forward pass модели
        embedded = self.embedding(input_batch)
        output, _ = self.lstm(embedded)
        output = self.dropout(output)
        logits = self.fc(output)
        logits = logits.reshape(-1, logits.size(-1))
        return logits

https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html

In [21]:
# Обучите модель здесь
model_1 = LanguageModel().to(device)

criterion = nn.CrossEntropyLoss(ignore_index=word2ind['<pad>'])
optimizer = torch.optim.Adam(model_1.parameters())

_, _, best_ppl_1 = train_model(
    model=model_1,
    train_dataloader=train_dataloader,
    eval_dataloader=eval_dataloader,
    optimizer=optimizer,
    criterion=criterion,
    num_epochs=5)

print('Лучшая eval perplexity (эксперимент 1):', best_ppl_1)

test_ppl_1 = evaluate(model_1, criterion, test_dataloader)
print('Test perplexity (эксперимент 1):', test_ppl_1)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


Epoch 1/5:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 1: train_loss=5.529, eval_ppl=160.613


Epoch 2/5:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 2: train_loss=4.932, eval_ppl=128.048


...

Epoch 3: train_loss=4.710, eval_ppl=114.345


Epoch 4/5:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 4: train_loss=4.556, eval_ppl=107.704


Epoch 5/5:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 5: train_loss=4.439, eval_ppl=104.617
Лучшая eval perplexity (эксперимент 1): 104.61708423926396
Test perplexity (эксперимент 1): 104.68761074617974


### Второй эксперимент

Попробуйте что-то поменять в модели или в пайплайне обучения, идеи для экспериментов можно подсмотреть выше.

https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html

In [22]:
# Проведите второй эксперимент
model_2 = LanguageModel().to(device)

criterion = nn.CrossEntropyLoss(ignore_index=word2ind['<pad>'])
optimizer = torch.optim.AdamW(model_2.parameters())

_, _, best_ppl_2 = train_model(
    model=model_2,
    train_dataloader=train_dataloader,
    eval_dataloader=eval_dataloader,
    optimizer=optimizer,
    criterion=criterion,
    num_epochs=5)

print('Лучшая eval perplexity (эксперимент 2):', best_ppl_2)

test_ppl_2 = evaluate(model_2, criterion, test_dataloader)
print('Test perplexity (эксперимент 2):', test_ppl_2)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


Epoch 1/5:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 1: train_loss=5.522, eval_ppl=160.848


Epoch 2/5:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 2: train_loss=4.932, eval_ppl=127.702


...

Epoch 3: train_loss=4.709, eval_ppl=113.907


Epoch 4/5:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 4: train_loss=4.556, eval_ppl=107.287


Epoch 5/5:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 5: train_loss=4.439, eval_ppl=104.006
Лучшая eval perplexity (эксперимент 2): 104.00648839218812
Test perplexity (эксперимент 2): 104.33281851714511


### Отчет

Опишите проведенные эксперименты. Сравните перплексии полученных моделей. Предложите идеи по улучшению качества моделей.

В 1 и 2 экспериментах рассматривалась одна и та же модель `1 слой LSTM, embedding_dim=128, hidden_size=256`  
Отличия в оптимизаторах **Adam** и **AdamW** (измененная L2-регуляризация)

**Результаты:** Adam добился метрики perplexity ~104.6, AdamW незначительно улучшил этот показатель

Идеи по улучшению:
- Эксперименты с гиперпараметрами, числом слоёв
- Расширение и чистка датасета